<a href="https://colab.research.google.com/github/ankit-3131/ai-frameworks/blob/main/chained_rag_on_webpages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U "langchain[google-genai]"

In [ ]:
import os
import getpass
from langchain.chat_models import init_chat_model
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('NEW_GEMINI')

model = init_chat_model("google_genai:gemini-3-flash-preview")

In [ ]:
#webscraping using loader
!pip install firecrawl-py
!pip install langchain-community

from langchain_community.document_loaders.firecrawl import FireCrawlLoader

In [ ]:
os.environ["FIRECRAWL_API"] = userdata.get('FIRECRAWL_API')
loader = FireCrawlLoader(
    api_key=userdata.get('FIRECRAWL_API'), url="https://leetcode.com/u/ankit_3131/", mode="scrape"
)

pages = []
for doc in loader.lazy_load():
    pages.append(doc)

In [ ]:
pages[0]

Document(metadata={'title': 'ankit_3131 - LeetCode Profile', 'description': "View ankit_3131's profile on LeetCode, the world's largest programming community.", 'url': 'https://leetcode.com/u/ankit_3131/', 'language': 'en', 'keywords': None, 'robots': 'index,follow', 'og_title': 'ankit_3131 - LeetCode Profile', 'og_description': "View ankit_3131's profile on LeetCode, the world's largest programming community.", 'og_url': None, 'og_image': 'https://leetcode.com/static/images/LeetCode_Sharing.png', 'og_audio': None, 'og_determiner': None, 'og_locale': 'en_US', 'og_locale_alternate': None, 'og_site_name': 'LeetCode', 'og_video': None, 'favicon': None, 'dc_terms_created': None, 'dc_date_created': None, 'dc_date': None, 'dc_terms_type': None, 'dc_type': None, 'dc_terms_audience': None, 'dc_terms_subject': None, 'dc_subject': None, 'dc_description': None, 'dc_terms_keywords': None, 'modified_time': None, 'published_time': None, 'article_tag': None, 'article_section': None, 'source_url': 'ht

In [ ]:
#splitters
!pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=30,
    chunk_overlap=5,
    length_function=len,
    is_separator_regex=False,
)
texts = text_splitter.create_documents([pages[0].page_content])
print(texts)


[Document(metadata={}, page_content='![Avatar](https://assets.leetc'), Document(metadata={}, page_content='leetcode.com/users/default_ava'), Document(metadata={}, page_content='t_avatar.jpg)'), Document(metadata={}, page_content='ankit\\_3131'), Document(metadata={}, page_content='![https://assets.leetcode.com'), Document(metadata={}, page_content='e.com/static_assets/others/lg2'), Document(metadata={}, page_content='s/lg2550.png](https://assets.l'), Document(metadata={}, page_content='ets.leetcode.com/static_assets'), Document(metadata={}, page_content='ssets/others/lg2550.png)'), Document(metadata={}, page_content='ankit\\_3131\n\nRank572,050\n\n1'), Document(metadata={}, page_content='1\n\nFollowing\n\n0\n\nFollowers'), Document(metadata={}, page_content='Community Stats\n\nViews\n\n0'), Document(metadata={}, page_content='0\n\nLast week0\n\nSolution\n\n0'), Document(metadata={}, page_content='0\n\nLast week0\n\nDiscuss\n\n0'), Document(metadata={}, page_content='0\n\nLast week0\n\n

In [ ]:
!pip install -qU  langchain-google-genai

In [ ]:
!pip install -qU langchain-qdrant

In [ ]:
!pip install -qU langchain-nvidia-ai-endpoints

In [ ]:
#vector store
import os
import shutil
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('NEW_GEMINI')

embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

qdrant_path = "/langchain_qdrant1"

if os.path.exists(qdrant_path):
    shutil.rmtree(qdrant_path)

client = QdrantClient(path=qdrant_path)

client.create_collection(
    collection_name="demoo_collection",
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="demoo_collection",
    embedding=embedding_model,
)

In [ ]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(texts))]

vector_store.add_documents(documents=texts, ids=uuids)

In [ ]:

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)
def process_llm_response(llm_response):
    cont=""
    for res in llm_response:
        cont+=res.page_content
    return cont

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
prompt = ChatPromptTemplate.from_messages([
    ("system", "You have to give short answers based on provided context only"),
    ("human", "Context: {context} Question:{question}")
])

runnable1 = RunnableLambda(process_llm_response)
parallel_chain = {
    "context": retriever | runnable1,
    "question": RunnablePassthrough()
}

chain = parallel_chain | prompt | model
q = "what was my contest rating"
print(chain.invoke(q))

content=[{'type': 'text', 'text': '1,713', 'extras': {'signature': 'EpgDCpUDAb4+9vvh4fIQ/J/FyDM8MOYdi9Bxu96ipE8jkDORDIoFem1ArZkuJ32eV/WbuXrVWE0DCRQgSUCVa8oQMSTxOet0X6hkBOqEIFr9OQkp7Pfit/eKwBVN3AszlFw16JIfJljnsV9RuUY+bHEmEuJKCPT9QcF8PbaNI2//9iSr9GCbSTKhspbRHNwOAwsAB9ufvm9DqzR1ep8SwXbM32fJMdZESYII+fHjwXhYOAIAcYh4CFzwM7pLb7ssZACbwQlzY4m452enZK8Y5KRTKSl1rBsMijwXsIBPAXRt0WlTi9GDvAgLBykazns6fsbzob0/uAxi36FmhpcR0hXpqdqA+iy8R7uIILYuk9IenVwgOJ4zoW5RGrg6TZYwvKKqKqGCjf8vsPuscP8thherQbnV3w52xAySGuY1mTIyTmvg/aEOw+13ULOnMwg1kKxmU4yTJi8Wc7J9QevWRSHURpv+rBwqmX78XuVsLOKOnbqLy+I7jE/yxVe++im+YDEbEqrAmZ3dybjX25VWso4y6XBwoXDUxysp'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019cc23c-22d6-71f1-a6f3-3a36913f5bc2-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 36, 'output_tokens': 122, 'total_tokens': 158, 'input_token_details': {'cache_read': 0}, 'output_t